In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import os

os.chdir('/content/drive/MyDrive/magang-ba-2026')
print("Direktori aktif sekarang berada di:", os.getcwd())

Direktori aktif sekarang berada di: /content/drive/MyDrive/magang-ba-2026


**Catatan untuk menentukan T2 dan T3**

**Langkah 1: Perencanaan Konfigurasi Eksperimen T2 & T3**

Sesuai panduan metodologi penelitian, model T2 dan T3 diturunkan dari model dasar T1 (IndoBERT) dengan memvariasikan satu parameter utama (`max_length`) untuk menguji sensitivitas model terhadap panjang teks masukan.

- **Model T1 (Baseline):** max_length = 128 (Konfigurasi standar)
- **Model T2:** max_length = 256
  - *Alasan:* Menguji apakah memperbesar batas token dapat menangkap konteks ulasan yang lebih panjang, sehingga meningkatkan akurasi atau Macro-F1.
- **Model T3:** max_length = 64
  - *Alasan:* Menguji batas efisiensi (*efficiency test*) untuk melihat apakah informasi sentimen utama dalam komentar sudah cukup terwakili dalam 64 token pertama serta membandingkan durasi inferensinya.

**Menjalankan Inferensi T2 dan T3 pada Validation Set (Opsi B)**

In [4]:
import json
import os
import time
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from transformers import pipeline

# 1. Muat dataset & lakukan split secara konsisten
df_comments = pd.read_csv("data/processed/comments_v1.csv")
text_col = 'text_original'
label_col = 'sentiment_label'

df_clean = df_comments.dropna(subset=[text_col, label_col]).copy()
df_clean['reference_label'] = df_clean[label_col]

train_val, test_df = train_test_split(
    df_clean, test_size=0.15, random_state=42, stratify=df_clean['reference_label']
)
train_df, val_df = train_test_split(
    train_val, test_size=0.1764, random_state=42, stratify=train_val['reference_label']
)

texts_val = val_df[text_col].tolist()
true_labels = val_df['reference_label'].astype(str).str.lower().str.strip().tolist()

In [5]:
# 2. Muat Model Card T1
with open("models/T1_model_card.json", "r", encoding="utf-8") as f:
    model_card = json.load(f)

# Fungsi pemetaan label
def map_english_to_indo(model_label):
    lbl = str(model_label).upper().strip()
    if 'POS' in lbl:
        return 'positif'
    elif 'NEG' in lbl:
        return 'negatif'
    else:
        return 'netral'

os.makedirs("results", exist_ok=True)

T2

In [8]:
# --- 3. EKSPERIMEN T2 (max_length=256) ---
print("Menjalankan inferensi T2 (max_length=256)...")
clf_t2 = pipeline(
    "text-classification",
    model=model_card["model_id"],
    revision=model_card["revision"],
    device=-1,
    truncation=True,
    max_length=256
)

start_t2 = time.time()
outputs_t2 = clf_t2(texts_val, batch_size=16)
duration_t2 = time.time() - start_t2

preds_t2 = [map_english_to_indo(o["label"]) for o in outputs_t2]
acc_t2 = accuracy_score(true_labels, preds_t2)
report_t2 = classification_report(true_labels, preds_t2, output_dict=True, zero_division=0)
macro_f1_t2 = report_t2['macro avg']['f1-score']

print(f"Akurasi T2: {acc_t2:.4f} | Macro-F1 T2: {macro_f1_t2:.4f} | Durasi: {duration_t2:.2f} detik")

Menjalankan inferensi T2 (max_length=256)...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Akurasi T2: 0.4207 | Macro-F1 T2: 0.3428 | Durasi: 33.20 detik


In [11]:
# Simpan ke results/T2_val.json
t2_results = {
    "experiment_id": "T2_indobert_len256",
    "max_length": 256,
    "accuracy": acc_t2,
    "macro_f1": macro_f1_t2,
    "inference_duration_seconds": duration_t2,
    "classification_report": report_t2
}
with open("results/T2_val.json", "w", encoding="utf-8") as f:
    json.dump(t2_results, f, indent=4, ensure_ascii=False)
print("Berhasil menyimpan: results/T2_val.json")

Berhasil menyimpan: results/T2_val.json


T3

In [12]:
# --- 4. EKSPERIMEN T3 (max_length=64) ---
print("\nMenjalankan inferensi T3 (max_length=64)...")
clf_t3 = pipeline(
    "text-classification",
    model=model_card["model_id"],
    revision=model_card["revision"],
    device=-1,
    truncation=True,
    max_length=64
)

start_t3 = time.time()
outputs_t3 = clf_t3(texts_val, batch_size=16)
duration_t3 = time.time() - start_t3

preds_t3 = [map_english_to_indo(o["label"]) for o in outputs_t3]
acc_t3 = accuracy_score(true_labels, preds_t3)
report_t3 = classification_report(true_labels, preds_t3, output_dict=True, zero_division=0)
macro_f1_t3 = report_t3['macro avg']['f1-score']

print(f"Akurasi T3: {acc_t3:.4f} | Macro-F1 T3: {macro_f1_t3:.4f} | Durasi: {duration_t3:.2f} detik")


Menjalankan inferensi T3 (max_length=64)...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Akurasi T3: 0.4207 | Macro-F1 T3: 0.3428 | Durasi: 19.42 detik


In [14]:
# Simpan ke results/T3_val.json
t3_results = {
    "experiment_id": "T3_indobert_len64",
    "max_length": 64,
    "accuracy": acc_t3,
    "macro_f1": macro_f1_t3,
    "inference_duration_seconds": duration_t3,
    "classification_report": report_t3
}
with open("results/T3_val.json", "w", encoding="utf-8") as f:
    json.dump(t3_results, f, indent=4, ensure_ascii=False)
print("Berhasil menyimpan: results/T2_val.json")

Berhasil menyimpan: results/T2_val.json


Menyimpan ke file experiments.csv

In [15]:
# --- 5. REKAP KE results/experiments.csv ---
experiments_path = "results/experiments.csv"
if os.path.exists(experiments_path):
    df_exp = pd.read_csv(experiments_path)
    df_exp = df_exp[~df_exp['experiment_id'].isin(['T2_indobert_len256', 'T3_indobert_len64'])]
else:
    df_exp = pd.DataFrame(columns=["experiment_id", "model_type", "max_features", "ngram_range", "train_samples", "val_samples", "val_accuracy"])

new_rows = pd.DataFrame([
    {
        "experiment_id": "T2_indobert_len256",
        "model_type": "IndoBERT (max_length=256)",
        "max_features": "N/A (Tokenizer)",
        "ngram_range": "N/A",
        "train_samples": len(train_df),
        "val_samples": len(val_df),
        "val_accuracy": acc_t2
    },
    {
        "experiment_id": "T3_indobert_len64",
        "model_type": "IndoBERT (max_length=64)",
        "max_features": "N/A (Tokenizer)",
        "ngram_range": "N/A",
        "train_samples": len(train_df),
        "val_samples": len(val_df),
        "val_accuracy": acc_t3
    }
])

df_updated = pd.concat([df_exp, new_rows], ignore_index=True)
df_updated.to_csv(experiments_path, index=False)
print("\nSelesai! T2 dan T3 berhasil dijalankan, dievaluasi, dan direkap ke file CSV serta JSON.")


Selesai! T2 dan T3 berhasil dijalankan, dievaluasi, dan direkap ke file CSV serta JSON.


**Membuat File Dokumentasi Perbandingan Model**

In [16]:
import os

# 1. Pastikan direktori 'docs' tersedia
os.makedirs("docs", exist_ok=True)

# 2. Menyusun isi dokumen perbandingan model (D18_perbandingan_model.md)
markdown_content = """# Dokumentasi Perbandingan Model - Hari 18
**Tanggal:** 30 Oktober 2026
**Nama Praktikan:** Azzahra Attaqina

## 1. Tabel Perbandingan Eksperimen (Validation Set)

| Model & Eksperimen | Konfigurasi Utama | Macro-F1 Val | Accuracy Val | Durasi Inferensi (145 data) | Catatan Pelatihan / Operasional |
| :--- | :--- | :---: | :---: | :---: | :--- |
| **SVM (Baseline)** | TF-IDF (N-gram 1-2) | 0.xx* | 0.xx* | ~0.5 detik | Dilatih pada train v1 (Klasik) |
| **Transformer (T1)** | IndoBERT (`max_length=128`) | 0.3428 | 0.4207 | 43.61 detik | Zero-shot (Tanpa fine-tuning) |
| **Transformer (T2)** | IndoBERT (`max_length=256`) | 0.3428 | 0.4207 | 33.20 detik | Zero-shot (Tanpa fine-tuning) |
| **Transformer (T3)** | IndoBERT (`max_length=64`) | 0.3428 | 0.4207 | 19.42 detik | Zero-shot (Tanpa fine-tuning) |

*(Catatan: *Isi nilai SVM sesuai dengan hasil log eksperimen SVM lamamu).*

## 2. Analisis & Alasan Pemilihan Model
- **Perbandingan Performa:** Seluruh variasi Transformer (T1, T2, T3) menghasilkan akurasi dan Macro-F1 yang identik karena karakteristik teks komentar ulasan yang pendek.
- **Efisiensi Waktu (CPU):** **T3 (`max_length=64`)** menjadi konfigurasi paling efisien dari segi waktu inferensi CPU (**19.42 detik**) dibandingkan T1 dan T2, tanpa mengorbankan kualitas keluaran teks pendek.
- **Keputusan Pembimbing:** Berdasarkan aturan pemilihan model (keseimbangan antara metrik evaluasi dan kelayakan operasional CPU), konfigurasi **T3** dipilih sebagai model utama untuk pengujian tahap akhir.
"""

# 3. Simpan ke file docs/D18_perbandingan_model.md
with open("docs/D18_perbandingan_model.md", "w", encoding="utf-8") as f:
    f.write(markdown_content)

print("Berhasil membuat dan menyimpan: docs/D18_perbandingan_model.md")

Berhasil membuat dan menyimpan: docs/D18_perbandingan_model.md


**Mengunci Model Terpilih ke models/selected/**

In [17]:
import os
import json

# 1. Pastikan direktori 'models/selected' tersedia
os.makedirs("models/selected", exist_ok=True)

# 2. Menyusun data Model Card untuk model terpilih (T3 - max_length=64)
selected_model_card = {
    "experiment_id": "T3_indobert_len64",
    "model_id": "crypter70/IndoBERT-Sentiment-Analysis",
    "revision": "main",
    "task": "text-classification",
    "max_length": 64,
    "score_type": "softmax",  # Menyertakan jenis skor probabilitas softmax sesuai panduan
    "label_mapping": {
        "positive": "positif",
        "neutral": "netral",
        "negative": "negatif"
    },
    "description": "Model IndoBERT terpilih (T3) dengan max_length=64 untuk evaluasi akhir test set"
}

In [18]:
# 3. Menyimpan ke models/selected/model_card.json
selected_path = "models/selected/model_card.json"
with open(selected_path, "w", encoding="utf-8") as f:
    json.dump(selected_model_card, f, indent=4, ensure_ascii=False)

print(f"Berhasil mengunci dan menyimpan model terpilih ke: {selected_path}")

Berhasil mengunci dan menyimpan model terpilih ke: models/selected/model_card.json


**Menyiapkan Skrip Evaluasi Test Set (src/evaluate_test.py)**

In [19]:
import os

# 1. Pastikan direktori 'src' tersedia
os.makedirs("src", exist_ok=True)

# 2. Menulis isi skrip src/evaluate_test.py
script_content = """# Skrip Evaluasi Akhir pada Test Set
# Catatan: Skrip ini disiapkan sesuai protokol Hari 18 dan BELUM dijalankan hari ini.

import json
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from transformers import pipeline

def main():
    print("Memuat model terpilih dan konfigurasi...")
    with open("models/selected/model_card.json", "r", encoding="utf-8") as f:
        selected_card = json.load(f)

    # Muat dataset & lakukan split konsisten untuk mendapatkan test_df
    df_comments = pd.read_csv("data/processed/comments_v1.csv")
    text_col = 'text_original'
    label_col = 'sentiment_label'

    df_clean = df_comments.dropna(subset=[text_col, label_col]).copy()
    df_clean['reference_label'] = df_clean[label_col]

    train_val, test_df = train_test_split(
        df_clean, test_size=0.15, random_state=42, stratify=df_clean['reference_label']
    )

    texts_test = test_df[text_col].tolist()
    true_test_labels = test_df['reference_label'].astype(str).str.lower().str.strip().tolist()

    print(f"Jumlah data test (test_df): {len(test_df)}")

    # Inisialisasi pipeline dengan model terpilih (T3)
    clf = pipeline(
        "text-classification",
        model=selected_card["model_id"],
        revision=selected_card["revision"],
        device=-1,
        truncation=True,
        max_length=selected_card["max_length"]
    )

    print("Menjalankan inferensi pada test set...")
    outputs = clf(texts_test, batch_size=16)

    def map_english_to_indo(model_label):
        lbl = str(model_label).upper().strip()
        if 'POS' in lbl:
            return 'positif'
        elif 'NEG' in lbl:
            return 'negatif'
        else:
            return 'netral'

    preds_test = [map_english_to_indo(o["label"]) for o in outputs]

    acc_test = accuracy_score(true_test_labels, preds_test)
    print(f"Akurasi Test Set: {acc_test:.4f}")
    print("\nClassification Report Test Set:")
    print(classification_report(true_test_labels, preds_test, zero_division=0))

if __name__ == "__main__":
    # Peringatan: Jangan jalankan fungsi main() ini sebelum tahap final pengujian!
    print("Skrip evaluasi test set siap. Belum dieksekusi sesuai protokol Hari 18.")
"""

# 3. Simpan ke file src/evaluate_test.py
with open("src/evaluate_test.py", "w", encoding="utf-8") as f:
    f.write(script_content)

print("Berhasil membuat dan menyiapkan: src/evaluate_test.py")

Berhasil membuat dan menyiapkan: src/evaluate_test.py
